# 02 · The robot world & an imitation-learning dataset

A robot VLM is only as good as its data. Before touching real hardware we build a **simulator** of the fetch task (`src/edge_vlm/sim/world.py`):

* 4 rooms (`living_room` where you sit, plus `kitchen`, `bedroom`, `office`), each with a distinct floor texture
* 20 object types (4 colors × 5 shapes), with **hard distractors** such as a *red ball* next to the *red cup* you asked for
* a 64×64 egocentric camera image
* 4 high-level skills: `navigate <room>`, `pick <color> <object>`, `return`, `handover`

Then we record a privileged **expert** solving thousands of episodes, which gives us supervised data for **behaviour cloning**.

> Later you can swap this toy world for **Isaac Sim** (photoreal, physics) or real teleoperation data. The agent and model code do not change, only the images get richer.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
from edge_vlm.sim import FetchWorld
from edge_vlm.domain import ROOMS

world = FetchWorld(seed=7)
print('command :', world.command)
print('target  :', world.target, 'in', world.target_room)
fig, axes = plt.subplots(1, 4, figsize=(12, 3.4))
for ax, room in zip(axes, ROOMS):
    ax.imshow(world.render(room)); ax.axis('off')
    ax.set_title(room + ('  <- target' if room == world.target_room else ''), fontsize=9)
    ax.text(0, 70, '\n'.join(world.visible_objects(room)), fontsize=7, va='top')
plt.tight_layout(); plt.show()

## The expert (teacher) policy

`expert_action` in `src/edge_vlm/agent.py` is ~10 lines of rules:

1. Holding something? Go back to the user (`return`), then `handover`.
2. Target visible in the current room? `pick` it.
3. User mentioned a room I haven't searched? `navigate` there.
4. Otherwise `navigate` to the next unsearched room.

It cheats: it reads the **ground-truth** object list. The VLM never gets that list. It has to learn rule 2 from **pixels**. Rules 1, 3 and 4 need **language** understanding (parse the command, read the state). That combination is exactly what a VLM is for.

In [ ]:
from edge_vlm.agent import ExpertPolicy, FetchAgent, evaluate_policy

world = FetchWorld(seed=7)
result = FetchAgent(ExpertPolicy()).run(world)
for r in result.trajectory:
    print(f"step {r['step']}: at {r['room']:12s} holding {str(r['holding']):12s} -> {r['output']:18s} | {r['result']}")
print('success:', result.success)
print('expert over 200 episodes:', evaluate_policy(ExpertPolicy(), n_episodes=200))

## Generating the dataset

Each **decision point** becomes one sample: `(camera image, prompt) → expert action`.

**Compounding errors and DART noise.** If we only record perfect demos, the model never sees states that come after its own mistakes, so one mistake can lead to a cascade. While recording, we sometimes *execute* a random detour (but still label the state with the expert's correct action). The model then learns how to recover. This is the idea behind DART/DAgger-style data collection, and it matters a lot on real robots.

In [ ]:
import time
from edge_vlm.data import generate_samples, action_distribution

t = time.time()
samples = generate_samples(500, seed=0, noise_prob=0.2)
print(f'{len(samples)} samples from 500 episodes in {time.time() - t:.1f}s')
print('label distribution:', action_distribution(samples))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, s in zip(axes.flat, samples[:8]):
    ax.imshow(s.image); ax.axis('off')
    p = s.prompt.replace(' | ', '\n')
    ax.set_title(f'{p}\n=> {s.action}', fontsize=7, loc='left')
plt.tight_layout(); plt.show()

## Why the image is essential: same text, different answer

Two states can have **identical prompts** and still need different actions, because the only difference is what the camera sees. A text-only model cannot solve this. That is our proof that the model must actually *look*.

In [ ]:
from collections import defaultdict
by_prompt = defaultdict(set)
for s in samples:
    by_prompt[s.prompt].add(s.action)
ambiguous = {p: a for p, a in by_prompt.items() if len(a) > 1}
print(f'{len(ambiguous)} of {len(by_prompt)} distinct prompts need the image to decide')
for p, a in list(ambiguous.items())[:3]:
    print(' ', p, '->', sorted(a))

## From toy data to real data

| Source | Realism | Cost | Use |
|---|---|---|---|
| This toy sim | low | free, instant | learn the pipeline, debug |
| **Isaac Sim** + Replicator | high (photoreal, physics) | GPU hours | domain-randomized training images, test before hardware |
| **Cosmos** world models | high | GPU hours | augment/transfer real or sim video (weather, lighting, textures) |
| Real teleop (e.g. LeRobot) | perfect | your time | final fine-tuning, closes the sim-to-real gap |

A common recipe is to pretrain on lots of sim, fine-tune on a few hundred real episodes, and evaluate on the real robot.

### Exercises
1. Set `p_wrong_hint=0.5` in `FetchWorld`. How does the average number of expert steps change?
2. Add a new object kind (for example `can`) in `domain.py` and a shape in `sim/world._draw`. Regenerate the data.
3. Add a fifth room. What changes in the prompt, tokenizer and expert?